In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    coalesce,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_restructuring_writeoff_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RestructuringWriteoff"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== it_restructures ==================

        it_restructures_extra_filters = None

        if it_restructures_extra_filters is not None:
            it_restructures_extra_filters.append(date_filter)
            it_restructures_final_filters = " AND ".join(it_restructures_extra_filters)
        else:
            it_restructures_final_filters = date_filter  # possible need change date

        it_restructures_selected_fields = [
            "RestructuringKey",
            "ReferenceDate",
            "RestructuringSystem",
            "PortfolioKey",
            "AgreementKey",
            "NewExposureKey",
            "OldExposureKey",
            "ProfitsId",
            "VatNumber",
            "CDI",
            "ProfitsId_PrimaryOwner",
            "VatNumber_PrimaryOwner",
            "CDI_PrimaryOwner",
            "InterestExposureKey",
            "InterestFreeExposureKey",
            "RestructuringAmount",
            "Writeoff_ExposureKey",
            "Writeoff_Amount",
            "SpreadPercentage",
            "RestructuringDate",
            "RestructuringProductKey",
            "Currency",
            "RestructuringTypeCode",
            "RestructuringType",
            "BusinessDate"
        ]

        it_restructures = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_it_restructures",
            it_restructures_final_filters,
            it_restructures_selected_fields,
        )
        #filter only restructures with writeoff
        it_restructures = it_restructures.filter(it_restructures.Writeoff_ExposureKey.isNotNull())

        # HashKeys
        writeoff_primary_key_fields = [
            it_restructures.RestructuringKey,
            lit("profits"),
        ]

        writeoff = (
            it_restructures.join(
                portfolio,
                ((portfolio["unit_code_profits"] == it_restructures["PortfolioKey"] ) & (portfolio["BusinessDate"] == it_restructures["BusinessDate"])),
                "left",
            )
            .withColumn(
                "RestructuringWriteoffId",
                sha2(concat_ws("", *writeoff_primary_key_fields), 256),
            )
            .withColumn(
                "RestructuringWriteoffKey",
                concat_ws(
                    ";",
                    it_restructures.NewExposureKey,
                    it_restructures.Writeoff_ExposureKey,
                )
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_profits = writeoff.select(
            writeoff.RestructuringWriteoffId.alias("RestructuringWriteoffId"),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            writeoff.RestructuringWriteoffKey.alias("RestructuringWriteoffKey"),
            it_restructures.NewExposureKey.alias("RestructuringExposureKey"),
            it_restructures.Writeoff_ExposureKey.alias("WriteOffExposureKey"),
            it_restructures.BusinessDate.alias("BusinessDate"),
        )

        # ================== Append ==================
        result_df_profits = (
            result_df_profits.distinct()
        )  # Eliminate duplicates in SourceSystem
        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_restructuring_writeoff_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)